# TWYD End-to-End: env + train + convert + eval
Train What You Deploy: Dense-LRC / CORE-LRC on ms-swift.
- plain: W=T Z_col^T only rH | dense/core: r*d_ff (reachability gap)
- Loss Eq.6-7: T^2 KL(T=40)+CE+0.2 aux | Init Alg.1 ridge Eq.4 + outlier-P Eq.5 (Qwen k=48)

In [ ]:
import os, sys, torch
ROOT='/home/chenwh/Papers/twyd'
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert os.path.isdir(ROOT) and os.path.isdir(ROOT+'/_upstream/ms-swift')
print('paths ok')

## 0. env
- swift env: /home/chenwh/miniconda3/envs/swift/bin/python | GPU bf16 | CPU can run secs 1-3
- _upstream/ read-only, only install: pip install -e _upstream/ms-swift + pip install -e .
- ModelScope default for model/dataset; HF needs --use_hf true

In [ ]:
%%bash
set -ex
PY=/home/chenwh/miniconda3/envs/swift/bin/python
$PY -c "import torch, transformers, swift; print('swift ok')"
cd /home/chenwh/Papers/twyd
$PY -m pip install -e ./_upstream/ms-swift -q 2>&1 | tail -n 2
$PY -m pip install -e . -q 2>&1 | tail -n 2
$PY -c "import twyd.plugin as P; print(sorted(P._TUNERS), sorted(P._LOSSES))"

## 1. checks (CPU, must-run)
pytest 14 passed + cpu_smoke tiny-Llama + calib synthetic

In [ ]:
%%bash
set -ex
cd /home/chenwh/Papers/twyd
/home/chenwh/miniconda3/envs/swift/bin/python -m pytest tests/ -q 2>&1 | tail -n 3
/home/chenwh/miniconda3/envs/swift/bin/python scripts/cpu_smoke.py --steps 10 2>&1 | tail -n 10
/home/chenwh/miniconda3/envs/swift/bin/python scripts/calib_ridge.py --synthetic 2>&1 | tail -n 2

### 1b. param count proves reachable gap (plain rH vs dense/core rd_ff)

In [ ]:
from twyd.config import TwydLRCConfig
from twyd.tuner import _build_factorized_student, _freeze_for_branch, _TwydWrapper
from tests.test_twyd_synthetic import _make_pair
for br in ['plain','dense','core']:
    t,s=_make_pair(24,64,12)
    cfg=TwydLRCConfig(branch=br, student_hidden=12)
    _build_factorized_student(s,t,cfg,branch=br)
    w=_TwydWrapper(s,t,cfg); _freeze_for_branch(w,cfg)
    n=sum(p.numel() for p in w.student.parameters() if p.requires_grad)
    print(br, n)  # plain 12288, dense/core 24576

## 2. train (swift pt main + standalone fallback)
--tuner_type dense_lrc|core_lrc|plain_lrc --loss_type lrc_distill --external_plugins twyd.plugin
--tuner_kwargs: teacher_model/student_hidden/kl_temperature=40/aux_weight=0.2/outlier_keep(Qwen 48, Llama 0)
matched (Table.13): same seed/data/order/budget, only col(T) vs full differs

In [ ]:
%%bash
cat /home/chenwh/Papers/twyd/scripts/train_dense.sh
echo '-------------------'
cat /home/chenwh/Papers/twyd/scripts/train_core.sh
# real run (Qwen0.5B smoke):
# STUDENT=Qwen/Qwen2.5-0.5B TEACHER=Qwen/Qwen2.5-0.5B DATASET=AI-ModelScope/alpaca-gpt4-data-zh#500 STUDENT_HIDDEN=448 bash scripts/train_dense.sh --output_dir outputs/dense-05b-smoke --seed 1234

### standalone fallback (no swift dep): KL+CE dense student

In [ ]:
%%bash
cat /home/chenwh/Papers/twyd/scripts/smoke_qwen05b.sh
echo '-------------------'
cat /home/chenwh/Papers/twyd/scripts/smoke_qwen3vl.sh
# bash scripts/smoke_qwen05b.sh

## 3. convert (merge to standard HF, zero infer cost)
- training auto merges via Adapter.save_pretrained -> merge_and_save
- manual: convert_ckpt.py --ckpt <swift out> --save <deploy>
- check: gate/up d_ff x r, down r x d_ff, no .z_col/.z_perp/basis_ leak

In [ ]:
from twyd.merge import merged_state_dict
from tests.test_twyd_synthetic import _make_pair, _wrap
for br in ['dense','core','plain']:
    t,s=_make_pair(24,64,12)
    w,_=_wrap(br,t,s,12)
    sd=merged_state_dict(w)
    leaks=[k for k in sd if '.z_col' in k or '.z_perp' in k or 'basis_' in k]
    print(br, 'leak:', leaks, 'n:', len(sd))
print('merge shapes OK')
# !python scripts/convert_ckpt.py --ckpt outputs/dense-05b-smoke/checkpoint-100 --save outputs/dense-05b-deploy

## 4. eval (swift eval + generation check)
- smoke: swift eval --eval_backend Native --eval_dataset gsm8k arc --eval_limit 10
- paper Avg9: 9-task 0-shot incl MMLU excl MathQA (lm-eval same protocol)
- short SFT then re-eval (Table.5 PT+SFT)

In [ ]:
%%bash
# CUDA_VISIBLE_DEVICES=0 swift eval --model outputs/dense-05b-deploy --eval_backend Native --infer_backend transformers --eval_limit 10 --eval_dataset gsm8k arc
# CUDA_VISIBLE_DEVICES=0 swift eval --model outputs/dense-05b-deploy --eval_backend Native --infer_backend transformers --eval_dataset mmlu arc hellaswag truthful_qa gsm8k
echo 'fill CKPT then uncomment above'

In [ ]:
CKPT='outputs/dense-05b-deploy'  # change to your deploy dir
print('set CKPT to run generation check')
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
# tok=AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
# m=AutoModelForCausalLM.from_pretrained(CKPT, dtype=torch.bfloat16, trust_remote_code=True).eval()
# ids=tok('Hello, my name is', return_tensors='pt').input_ids
# print(tok.decode(m.generate(ids, max_new_tokens=20)[0]))

## 5. troubleshooting
| symptom | cause | fix |
|---|---|---|
| KL=None / CE-only | swift strips inputs | wrapper _twyd_last_inputs auto-fallback, or --callbacks twyd.plugin.TwydInputCacheCallback |
| aux always 0 | no hidden_states | use_aux=false KL-only first (Table.12 stripped recipe) |
| Qwen align collapse | massive-act channels | tuner_kwargs outlier_keep=48 (Llama 0) |
| dense params=rH | old CoreLinear branch | fixed to nn.Linear d_ff x r, verify with sec 1b |